# Preparación de los datos espaciales agrícolas

**Autor:** Andres Felipe Sierra

**Objetivo:** Análisis y preparación de los datos geoespaciales agrícolas de UPRA para la conmutación y completar la base del proyecto de grado.

**Apreciaciones:** Este script tiene la implementación de CRISP-DM con los respectivos ciclos siendo: Bussiness understanding, Data understanding, Data preparation, Modeling, Evaluation y Deployment. 

## Bussiness understanding

Preparar los datos geoespaciales Frontera agrícola, Identificación de las áreas con restricciones al desarrollo de las actividades agropecuarias en Colombia, Aptitud arroz y Aptitud maíz para incluirlos en la base de datos proyectada a un modelo ML de riesgo agrícola por inundación y deslizamiento.

### Librerias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re

import IPython.display 
import geopandas as gpd
import fiona
import pyogrio

### Funciones

#### EVA

In [2]:
# Función para ver información del dataset
def info_data(dat):
    print("El dataset consta de ",dat.shape, " (FILAS/COLUMNAS)\n")
    print("Información del dataset: \n")
    dat.info()

In [3]:
# Función para la revisión de los valores duplicados y droppearlos
def datos_duplicados(dat):
    print('Tamaño del dataset original: ', dat.shape)
    da = dat.drop_duplicates(keep= 'first', inplace=False)
    print('Tamaño del dataset sin duplicados: ',da.shape)
    return da

In [4]:
# Función para la revisión de los valores nulos
def datos_nulos(dat, solo_nulos=False, ordenar=True):
    nulos = dat.isna().sum()
    pct = (nulos / len(dat) * 100).round(2)

    out = pd.DataFrame({
        "nulos": nulos,
        "% nulos": pct,
        "tipo": dat.dtypes.astype(str)
    })

    if solo_nulos:
        out = out[out["nulos"] > 0]

    if ordenar:
        out = out.sort_values("nulos", ascending=False)

    display(out)

In [5]:
# Función de asegurar que el código del municipio sea string de 5 dígitos
def cod_municipio_5dig(dat, col):
    dat[col] = dat[col].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5) 
    return dat

In [6]:
# Función de filtrar registros
def filtrar_cultivos(dat,col):
    df = dat[dat['Cultivo'].isin(col)].copy()
    return df

In [7]:
# Función para pivotar tabla
def pivotar_tabla_cul(dat):
    df = pd.pivot_table(
        dat, 
        values=['Área cosechada (ha)', 'Producción (t)'], 
        index=['Código Dane municipio', 'Año'], 
        columns=['Cultivo'], 
        aggfunc='sum',
        fill_value=0
    )
    return df

#### GEOPANDAS

In [8]:
# Función para leer geopandas
def capa_geopandas(rut):
    df = fiona.listlayers(rut)
    print("Capas disponibles en la GDB:", df)
    return df

In [9]:
def procesar_aptitud(ruta_shp, nombre_cultivo):
    # Se lee el shapefile
    gdf = gpd.read_file(ruta_shp, engine='pyogrio')
    
    # Imprimimos las columnas para ver como se llama la variable de aptitud y si trae código DANE
    columnas = gdf.columns.str.lower().tolist()
    print(f"Columnas detectadas en {nombre_cultivo}: {columnas}")
    return gdf

In [10]:
# Función para mostrar los datos sin la variable geometrica
def info_sin_geo(dat):
    # Mostramos una muestra de los datos del maíz
    print("\nMuestra de datos sin GEO:")
    columnas_sin_geo_maiz = [col for col in dat.columns if col != 'geometry']
    print(dat[columnas_sin_geo_maiz].head())

In [11]:
# Función para limpiar la aptitud y el código DIVIPOLA
def consolidar_aptitud_municipal(dat, nombre_cultivo):
    # Limpieza de DIVIPOLA
    dat['cod_dane_m'] = dat['cod_dane_m'].astype(str).str.zfill(5)
    
    # Agrupación y Pivotaje
    # Sumamos las hectáreas por municipio y por nivel de aptitud
    df_pivot = dat.pivot_table(
        index='cod_dane_m', 
        columns='aptitud', 
        values='area_ha', 
        aggfunc='sum', 
        fill_value=0
    ).reset_index()
    
    # Renombrarmos las columnas para identificar el cultivo
    # Tipo: 'Aptitud alta' -> 'maiz_aptitud_alta_ha'
    columnas_aptitud = ['Aptitud alta', 'Aptitud baja', 'Aptitud media', 'Exclusión legal', 'No apta']
    for col in df_pivot.columns:
        if col in columnas_aptitud:
            nuevo_nombre = f"{nombre_cultivo.lower()}_{col.lower().replace(' ', '_')}_ha"
            df_pivot = df_pivot.rename(columns={col: nuevo_nombre})
            
    return df_pivot

#### Resultados de la prepracaión

In [12]:
def limpiar_divipola(serie):
    # Extrae exclusivamente los números, elimina decimales (.0) y rellena con 5 ceros
    return serie.astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)

In [13]:
def normalizar_nombre_columna(col):
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    col = col.replace(" ", "_").replace(".", "_")
    col = re.sub(r"[^0-9a-zA-Z_]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col.lower()

## Data understanding

Se verifica que el dataset funcione y se comporte de manera funcional para su extracción y manipulación.

Lo que se realizó fue extraer los datos de la entidad nacional UPRA (Unidad de Planificación Rural Agropecuaria) con la herramiento SIPRA basandonos en las fronteras territoriales de los municipios y cultivos registrados.

El modo en el que se realizará es por dataset para no mezclar contenido.

### EVA

#### Exploración de datos

In [14]:
# Cargar la base agrícola EVA
df_eva = pd.read_excel('../Data/CULTIVOS/20250617_BaseAgricola20192024.xlsx', skiprows=7)

In [15]:
info_data(df_eva)

El dataset consta de  (141073, 18)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 141073 entries, 0 to 141072
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Código Dane departamento       141073 non-null  int64  
 1   Departamento                   141073 non-null  str    
 2   Código Dane municipio          141073 non-null  int64  
 3   Municipio                      141073 non-null  str    
 4   Desagregación cultivo          141073 non-null  str    
 5   Cultivo                        141073 non-null  str    
 6   Ciclo del cultivo              141073 non-null  str    
 7   Grupo cultivo                  141073 non-null  str    
 8   Subgrupo                       141073 non-null  str    
 9   Año                            141073 non-null  int64  
 10  Periodo                        141073 non-null  str    
 11  Área sembrada (ha)   

Buscamos si hay datos duplicados

In [16]:
# Datos duplicados
datos_duplicados(df_eva)

Tamaño del dataset original:  (141073, 18)
Tamaño del dataset sin duplicados:  (141073, 18)


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Desagregación cultivo,Cultivo,Ciclo del cultivo,Grupo cultivo,Subgrupo,Año,Periodo,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha),Nombre científico del cultivo,Código del cultivo,Estado físico del cultivo
0,5,Antioquia,5001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2019,2019,24.00,23.00,138.00,6.0,Persea americana,2040299,En fresco
1,5,Antioquia,5001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2020,2020,8.52,3.52,21.12,6.0,Persea americana,2040299,En fresco
2,5,Antioquia,5001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2021,2021,8.52,4.52,27.12,6.0,Persea americana,2040299,En fresco
3,5,Antioquia,5001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2022,2022,17.17,8.52,51.12,6.0,Persea americana,2040299,En fresco
4,5,Antioquia,5001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2023,2023,14.97,8.52,51.12,6.0,Persea americana,2040299,En fresco
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141068,99,Vichada,99773,Cumaribo,Yuca consumo en fresco,Yuca,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2020,2020A,450.00,400.00,4000.00,10.0,Manihot esculenta,1081001,En fresco
141069,99,Vichada,99773,Cumaribo,Yuca consumo en fresco,Yuca,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2021,2021A,380.00,380.00,3420.00,9.0,Manihot esculenta,1081001,En fresco
141070,99,Vichada,99773,Cumaribo,Yuca consumo en fresco,Yuca,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2022,2022A,350.00,380.00,3800.00,10.0,Manihot esculenta,1081001,En fresco
141071,99,Vichada,99773,Cumaribo,Yuca consumo en fresco,Yuca,Transitorio,Raíces y tubérculos,Raíces y tubérculos,2023,2023A,400.00,350.00,3500.00,10.0,Manihot esculenta,1081001,En fresco


No hay datos duplicados, lo que nos da paso a buscar is hay registros nulos

In [17]:
datos_nulos(df_eva)

,nulos,% nulos,tipo
Código Dane departamento,0,0.0,int64
Departamento,0,0.0,str
Código Dane municipio,0,0.0,int64
Municipio,0,0.0,str
Desagregación cultivo,0,0.0,str
Cultivo,0,0.0,str
Ciclo del cultivo,0,0.0,str
Grupo cultivo,0,0.0,str
Subgrupo,0,0.0,str
Año,0,0.0,int64


No posee datos nulos lo que da buena señal de la calidad del dataset.

#### Data preparation

Aquí se preparan los datos para poder descargarlos y hacer la respectiva unión.

In [18]:
# Asegurar que el código del municipio sea string de 5 dígitos
df_eva = cod_municipio_5dig(df_eva,'Código Dane municipio')

In [19]:
# Filtramos los datos
colum = ['Arroz', 'Maíz']
df_eva_filtrado = filtrar_cultivos(df_eva,colum)

In [20]:
# Se crea el pivote
df_exposicion_eva = pivotar_tabla_cul(df_eva_filtrado)

In [21]:
# Aplanar los niveles de las columnas resultantes
df_exposicion_eva.columns = [f"{col[1]}_{col[0]}".replace(' ', '_').lower() for col in df_exposicion_eva.columns]
df_exposicion_eva = df_exposicion_eva.reset_index()

In [22]:
info_data(df_exposicion_eva)

El dataset consta de  (6141, 6)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 6141 entries, 0 to 6140
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Código Dane municipio      6141 non-null   str    
 1   Año                        6141 non-null   int64  
 2   arroz_producción_(t)       6141 non-null   float64
 3   maíz_producción_(t)        6141 non-null   float64
 4   arroz_área_cosechada_(ha)  6141 non-null   float64
 5   maíz_área_cosechada_(ha)   6141 non-null   float64
dtypes: float64(4), int64(1), str(1)
memory usage: 318.7 KB


Ya con los datos filtrados, se elimina la variable df_eva para liberar espacio, por otro lado, con los registros extraídos se preparán para la unión con el dataset de la UNGRD.

In [23]:
del df_eva

### Restricciones

#### Exploración de datos

In [24]:
# Se extrae las capas del archivo .gdb
ruta = '../Data/CULTIVOS/Restricciones_FA_Jun2025.gdb/a0000000b.gdbtable'
capas_gdb = capa_geopandas(ruta)

print("Las capas disponibles dentro de esta GDB son:")
for capa in capas_gdb:
    print(f"- {capa}")

Capas disponibles en la GDB: ['Restricciones_FA_Jun2025']
Las capas disponibles dentro de esta GDB son:
- Restricciones_FA_Jun2025


In [25]:
# Dado a que se encuentra la capa, entramos en la misma para ver su contenido
nombre_capa = 'Restricciones_FA_Jun2025' 

# Leemos la capa con el motor pyogrio
gdf_restricciones = gpd.read_file(ruta, layer=nombre_capa, engine='pyogrio')

# Veamos qué columnas (variables) trae este mapa
print("Columnas disponibles:")
print(gdf_restricciones.columns.tolist())

# Veamos una muestra de las primeras 5 filas para entender los datos
print("\nMuestra de los datos:")
# Mostramos todas las columnas excepto la geometría (para que sea más fácil de leer en pantalla)
columnas_sin_geo = [col for col in gdf_restricciones.columns if col != 'geometry']
print(gdf_restricciones[columnas_sin_geo].head())

c:\Python\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Columnas disponibles:
['municipio', 'departamen', 'cod_depart', 'cod_dane_mpio', 'zona_restriccion', 'cat_restriccion', 'consecutivo', 'area_ha', 'Shape_Length', 'Shape_Area', 'geometry']

Muestra de los datos:
  municipio departamen cod_depart cod_dane_mpio  zona_restriccion  \
0    Suaita  Santander         68         68770  Zona restricción   
1    Suaita  Santander         68         68770  Zona restricción   
2    Suaita  Santander         68         68770  Zona restricción   
3    Suaita  Santander         68         68770  Zona restricción   
4    Suaita  Santander         68         68770  Zona restricción   

                            cat_restriccion  consecutivo   area_ha  \
0  Restricciones acuerdo cero deforestación      1855020  0.003645   
1  Restricciones acuerdo cero deforestación      1855021  0.092789   
2  Restricciones acuerdo cero deforestación      1855022  0.037256   
3  Restricciones acuerdo cero deforestación      1855023  0.009085   
4  Restricciones acuerdo

In [26]:
info_data(gdf_restricciones)

El dataset consta de  (2155920, 11)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2155920 entries, 0 to 2155919
Data columns (total 11 columns):
 #   Column            Dtype   
---  ------            -----   
 0   municipio         str     
 1   departamen        str     
 2   cod_depart        str     
 3   cod_dane_mpio     str     
 4   zona_restriccion  str     
 5   cat_restriccion   str     
 6   consecutivo       int32   
 7   area_ha           float64 
 8   Shape_Length      float64 
 9   Shape_Area        float64 
 10  geometry          geometry
dtypes: float64(3), geometry(1), int32(1), str(6)
memory usage: 351.1 MB


In [27]:
datos_duplicados(gdf_restricciones)

Tamaño del dataset original:  (2155920, 11)
Tamaño del dataset sin duplicados:  (2155920, 11)


,municipio,departamen,cod_depart,cod_dane_mpio,zona_restriccion,cat_restriccion,consecutivo,area_ha,Shape_Length,Shape_Area,geometry
0,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855020,0.003645,38.801689,36.452880,"MULTIPOLYGON (((4974271.166 2233033.926, 49742..."
1,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855021,0.092789,121.848844,927.894746,"MULTIPOLYGON (((4974486.051 2233063.7, 4974485..."
2,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855022,0.037256,112.671833,372.556596,"MULTIPOLYGON (((4974271.226 2233064.13, 497427..."
3,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855023,0.009085,104.831231,90.851794,"MULTIPOLYGON (((4974218.904 2233032.964, 49742..."
4,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855024,0.000790,97.105652,7.899613,"MULTIPOLYGON (((4974148.494 2233077.013, 49741..."
...,...,...,...,...,...,...,...,...,...,...,...
2155915,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855015,0.144365,898.198448,1443.654917,"MULTIPOLYGON (((4974152.33 2232709.286, 497414..."
2155916,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855016,0.961457,641.352727,9614.571039,"MULTIPOLYGON (((4973843.094 2232892.56, 497385..."
2155917,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855017,0.001300,40.034499,12.997712,"MULTIPOLYGON (((4973709.136 2233005.731, 49737..."
2155918,Suaita,Santander,68,68770,Zona restricción,Restricciones acuerdo cero deforestación,1855018,0.003878,65.004613,38.781216,"MULTIPOLYGON (((4973766.012 2233010.724, 49737..."


Se puede observar que el dataset tiene un total de 215920 registros con once columnas, además no posee datos duplicados.

El análisis anterior da píe a su preparación y extracción de datos de importancia para el proyecto.

#### Preaparación de datos

In [28]:
# Listar todos los tipos únicos de restricciones que la UPRA incluyó
categorias_unicas = gdf_restricciones['cat_restriccion'].unique()

print("Las categorías de restricción en Colombia son:")
for cat in categorias_unicas:
    print(f"- {cat}")

Las categorías de restricción en Colombia son:
- Restricciones acuerdo cero deforestación
- Restricciones legales
- Restricciones técnicas (Áreas no agropecuarias)


In [29]:
# Sumamos toda el área restringida por municipio (sin importar el tipo)
df_exclusiones_muni = gdf_restricciones.groupby('cod_dane_mpio')['area_ha'].sum().reset_index()

df_exclusiones_muni = df_exclusiones_muni.rename(columns={'area_ha': 'area_excluida_legal_ha'})
df_exclusiones_muni['cod_dane_mpio'] = df_exclusiones_muni['cod_dane_mpio'].astype(str).str.zfill(5)

print(df_exclusiones_muni.head())

  cod_dane_mpio  area_excluida_legal_ha
0         00000            28720.222862
1         05001            26621.389294
2         05002            15755.487456
3         05004            28270.434266
4         05021             4025.909961


La base de datos de restricciones no tiene los registros que se esperaban, lo que se decide extraer algunos datos para reforzar los datos de la UNGRD, además se pretende extraer otra base de datos con el mismo nombre, pero condicional para buscar si este ofrece registros de restricción en cuanto a inundaciones o algo parecido.

### Restricción condicional

#### Exploración de datos

In [30]:
#Cargar shapefile de municipios (DANE) y el de UPRA
#gdf_no_condiconada = gpd.read_file('../Data/CULTIVOS/Frontera_Agr_Cond_NCon_Jun2025/Frontera_Agr_Cond_NCon_Jun2025.shp')

In [31]:
#info_data(gdf_no_condiconada)

In [32]:
#gdf_no_condiconada.head()

In [33]:
#gdf_no_condiconada['tipo_front'].unique()

Este dataset tampoco ofrece los datos que se prentendían encontrar, por ende pasamos de este y utilizamos el anterior para tener información de los cultivos.

### Aptitud arroz

#### Exploración de datos

In [34]:
# Comenzamos la exploracion de los registros de arroz
ruta = '../Data/CULTIVOS/aptitud_arrozsecano_dic2019'
gdf_arroz = procesar_aptitud(ruta, "Arroz")

Columnas detectadas en Arroz: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m', 'gridcode', 'aptitud', 'area_ha', 'consecutiv', 'st_area_sh', 'st_length_', 'shape_leng', 'shape_area', 'geometry']


In [35]:
info_data(gdf_arroz)

El dataset consta de  (108163, 13)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 108163 entries, 0 to 108162
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   municipio   108163 non-null  str     
 1   departamen  108163 non-null  str     
 2   cod_depart  108163 non-null  str     
 3   cod_dane_m  108163 non-null  str     
 4   gridcode    108163 non-null  int64   
 5   aptitud     108163 non-null  str     
 6   area_ha     108163 non-null  float64 
 7   consecutiv  108163 non-null  int64   
 8   st_area_sh  108163 non-null  float64 
 9   st_length_  108163 non-null  float64 
 10  shape_Leng  108163 non-null  float64 
 11  shape_Area  108163 non-null  float64 
 12  geometry    108163 non-null  geometry
dtypes: float64(5), geometry(1), int64(2), str(5)
memory usage: 14.3 MB


In [36]:
# Vemos los datos sin la variable GEO
info_sin_geo(gdf_arroz)


Muestra de datos sin GEO:
   municipio          departamen cod_depart cod_dane_m  gridcode  \
0      Amagá           Antioquia         05      05030         8   
1  Aquitania              Boyacá         15      15047         0   
2  Arauquita              Arauca         81      81065         8   
3  Arboledas  Norte de Santander         54      54051         8   
4  Barbacoas              Nariño         52      52079         8   

           aptitud       area_ha  consecutiv    st_area_sh     st_length_  \
0  Exclusión legal    295.273172        1644  2.952732e+06   15794.905803   
1          No apta     77.674130        2072  7.767413e+05    7188.017906   
2  Exclusión legal    487.852249        3488  4.878522e+06  162681.132758   
3  Exclusión legal  21896.734933        3533  2.189673e+08  180885.849684   
4  Exclusión legal      0.000076        5122  7.625077e-01      61.554219   

      shape_Leng    shape_Area  
0   15794.905803  2.952732e+06  
1    7188.017906  7.767413e+05  
2 

In [37]:
gdf_arroz['aptitud'].unique()

<ArrowStringArray>
['Exclusión legal', 'No apta', 'Aptitud media', 'Aptitud baja',
 'Aptitud alta']
Length: 5, dtype: str

In [38]:
df_arroz_final = consolidar_aptitud_municipal(gdf_arroz, "Arroz")

### Aptitud maíz

#### Exploración de datos

In [39]:
# Comenzamos la exploracion de los registros de arroz
ruta = '../Data/CULTIVOS/aptitud_Maiz_CalidoS1_Dic2019'
gdf_maiz = procesar_aptitud(ruta, "Maíz")

Columnas detectadas en Maíz: ['municipio', 'departamen', 'cod_depart', 'cod_dane_m', 'gridcode', 'aptitud', 'area_ha', 'consecutiv', 'shape_leng', 'shape_area', 'geometry']


In [40]:
info_data(gdf_maiz)

El dataset consta de  (151733, 11)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 151733 entries, 0 to 151732
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   municipio   151733 non-null  str     
 1   departamen  151733 non-null  str     
 2   cod_depart  151733 non-null  str     
 3   cod_dane_m  151733 non-null  str     
 4   gridcode    151733 non-null  int64   
 5   aptitud     151733 non-null  str     
 6   area_ha     151733 non-null  float64 
 7   consecutiv  151733 non-null  int64   
 8   shape_Leng  151733 non-null  float64 
 9   shape_Area  151733 non-null  float64 
 10  geometry    151733 non-null  geometry
dtypes: float64(3), geometry(1), int64(2), str(5)
memory usage: 17.8 MB


In [41]:
info_sin_geo(gdf_maiz)


Muestra de datos sin GEO:
   municipio departamen cod_depart cod_dane_m  gridcode          aptitud  \
0   Cumaribo    Vichada         99      99773         0          No apta   
1  Abejorral  Antioquia         05      05002         8  Exclusión legal   
2   Cumaribo    Vichada         99      99773         1     Aptitud baja   
3    Aguadas     Caldas         17      17013         0          No apta   
4    Aguadas     Caldas         17      17013         8  Exclusión legal   

       area_ha  consecutiv     shape_Leng    shape_Area  
0  3871.071455          37  402228.870817  3.871071e+07  
1  1347.163069          65   16853.596009  1.347163e+07  
2  1390.995036         101   74709.768431  1.390995e+07  
3     0.001547         721      23.397485  1.546630e+01  
4     0.037760         829      89.638217  3.775967e+02  


In [42]:
gdf_maiz['aptitud'].unique()

<ArrowStringArray>
['No apta', 'Exclusión legal', 'Aptitud baja', 'Aptitud media',
 'Aptitud alta']
Length: 5, dtype: str

In [43]:
df_maiz_final = consolidar_aptitud_municipal(gdf_maiz, "Maiz")

### Frontera agricola

In [44]:
#Cargar shapefile
gdf_frontera_agricola = gpd.read_file('../Data/CULTIVOS/Frontera_Agricola_Jun2025/Frontera_Agricola_Jun2025.shp')

In [45]:
info_data(gdf_frontera_agricola)

El dataset consta de  (92384, 10)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 92384 entries, 0 to 92383
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   municipio   92384 non-null  str     
 1   departamen  92384 non-null  str     
 2   cod_depart  92384 non-null  str     
 3   cod_dane_m  92384 non-null  str     
 4   elemento    92384 non-null  str     
 5   area_ha     92384 non-null  float64 
 6   consecutiv  92384 non-null  int64   
 7   shape_Leng  92384 non-null  float64 
 8   shape_Area  92384 non-null  float64 
 9   geometry    92384 non-null  geometry
dtypes: float64(3), geometry(1), int64(1), str(5)
memory usage: 11.6 MB


In [46]:
gdf_frontera_agricola.head()

,municipio,departamen,cod_depart,cod_dane_m,elemento,area_ha,consecutiv,shape_Leng,shape_Area,geometry
0,Abejorral,Antioquia,05,05002,Frontera agrícola nacional,0.009605,1,80.532327,96.051171,"POLYGON ((4728922.601 2186655.289, 4728893.316..."
1,Abejorral,Antioquia,05,05002,Frontera agrícola nacional,0.028754,2,82.426583,287.542777,"POLYGON ((4728853.652 2186687.113, 4728853.627..."
2,Abejorral,Antioquia,05,05002,Frontera agrícola nacional,0.027532,3,80.196560,275.323549,"POLYGON ((4721544.859 2187041.553, 4721544.858..."
3,Abejorral,Antioquia,05,05002,Frontera agrícola nacional,0.031979,4,86.430025,319.788154,"POLYGON ((4721514.213 2187077.363, 4721514.199..."
4,Abejorral,Antioquia,05,05002,Frontera agrícola nacional,0.083272,5,152.425985,832.715962,"POLYGON ((4725155.377 2188701.211, 4725160.194..."


In [47]:
# Aseguramos el DIVIPOLA a 5 dígitos
gdf_frontera_agricola['cod_dane_clean'] = gdf_frontera_agricola['cod_dane_m'].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)

# Agrupamos por municipio sumando el área (area_ha)
# Usualmente la columna 'elemento' dice "Frontera agrícola nacional" en todos los registros
df_frontera_muni = gdf_frontera_agricola.groupby('cod_dane_clean')['area_ha'].sum().reset_index()

# Renombramos para el modelo
df_frontera_muni = df_frontera_muni.rename(columns={'area_ha': 'frontera_agricola_total_ha'})

### Resultado de la preparación

El proceso que se llevo a cabo permite la filtración geoespacial de los cultivos, en este caso dos cultivos particularmente para no extender el la complejidad de dataset, además de extraer las fronteras y valor de los mismos.

La finalidad es la unión de estos datos junto al conjunto de datos UNGRD.

In [48]:
# Unimos ambos resultados en una sola tabla de aptitud municipal
df_aptitud_total = pd.merge(df_maiz_final, df_arroz_final, on='cod_dane_m', how='outer').fillna(0)

print("Tabla de Aptitud Municipal consolidada:")
print(df_aptitud_total.head())

Tabla de Aptitud Municipal consolidada:
aptitud cod_dane_m  maiz_aptitud_alta_ha  maiz_aptitud_baja_ha  \
0            05001            215.488027              0.000000   
1            05002            176.093771            851.356037   
2            05004              0.000000              0.000000   
3            05021              0.000000            675.555925   
4            05030            215.245497            140.969829   

aptitud  maiz_aptitud_media_ha  maiz_exclusión_legal_ha  maiz_no_apta_ha  \
0                   250.814122             16303.947021     20118.949458   
1                  1527.369754              2491.432600     46013.142773   
2                     0.000000             29118.669671       166.211817   
3                   224.753719              3471.434839      8886.723746   
4                   529.579400               394.196835      7007.021034   

aptitud  arroz_aptitud_alta_ha  arroz_aptitud_baja_ha  arroz_aptitud_media_ha  \
0                     0.0

In [49]:
info_data(df_aptitud_total)

El dataset consta de  (1122, 11)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cod_dane_m                1122 non-null   str    
 1   maiz_aptitud_alta_ha      1122 non-null   float64
 2   maiz_aptitud_baja_ha      1122 non-null   float64
 3   maiz_aptitud_media_ha     1122 non-null   float64
 4   maiz_exclusión_legal_ha   1122 non-null   float64
 5   maiz_no_apta_ha           1122 non-null   float64
 6   arroz_aptitud_alta_ha     1122 non-null   float64
 7   arroz_aptitud_baja_ha     1122 non-null   float64
 8   arroz_aptitud_media_ha    1122 non-null   float64
 9   arroz_exclusión_legal_ha  1122 non-null   float64
 10  arroz_no_apta_ha          1122 non-null   float64
dtypes: float64(10), str(1)
memory usage: 102.2 KB


#### Unión del dataset

##### Cargue de los datos UNGRD

Debido a problemas de cocatenación o unión de los dataset se decide realizar nuevamente una limpieza de los datosde DIVIPOLA.

In [50]:
# Cargar la base agrícola EVA
df_ungrd= pd.read_excel('../Data_preparada/datos_inundaciones_v2.xlsx')

In [51]:
df_ungrd['DIVIPOLA'] = df_ungrd['DIVIPOLA'].astype(str)

In [52]:
# Nos aseguramos que FECHA sea datetime
df_ungrd['FECHA'] = pd.to_datetime(df_ungrd['FECHA'], errors='coerce')
df_ungrd['year'] = df_ungrd['FECHA'].dt.year
df_ungrd['month'] = df_ungrd['FECHA'].dt.month
df_ungrd['ym'] = df_ungrd['FECHA'].dt.to_period('M') # Formato 'YYYY-MM'

In [53]:
# Limpiamos la base principal
df_ungrd['DIVIPOLA_clean'] = limpiar_divipola(df_ungrd['DIVIPOLA'])
df_ungrd['year_clean'] = df_ungrd['year'].astype(int)

In [54]:
info_data(df_ungrd)

El dataset consta de  (1192, 30)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [55]:
# Limpiamos la base de Aptitud (UPRA)
df_aptitud_total['cod_dane_clean'] = limpiar_divipola(df_aptitud_total['cod_dane_m'])

# Limpiamos la base de Producción (EVA)
df_exposicion_eva['cod_dane_clean'] = limpiar_divipola(df_exposicion_eva['Código Dane municipio'])

In [56]:
# Aseguramos que el año sea un número entero limpio
df_exposicion_eva['Año_clean'] = df_exposicion_eva['Año'].fillna(0).astype(int)

##### Unión de los datos

###### Unión de los datos de ungrd y aptitudes del maiz y arroz

In [57]:
# 2.1 Unión Estática: Eventos + Aptitudes (UPRA)
df_master = df_ungrd.merge(
    df_aptitud_total, 
    left_on='DIVIPOLA_clean', 
    right_on='cod_dane_clean', 
    how='left'
)

In [58]:
info_data(df_master)

El dataset consta de  (1192, 42)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 42 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [59]:
df_master.head()

,FECHA,DIVIPOLA,DEPARTAMENTO,MUNICIPIO,EVENTO,CODIGO EMERGENCIA,PERSONAS,FAMILIAS,HECTAREAS,VIV.DESTRU.,...,maiz_aptitud_baja_ha,maiz_aptitud_media_ha,maiz_exclusión_legal_ha,maiz_no_apta_ha,arroz_aptitud_alta_ha,arroz_aptitud_baja_ha,arroz_aptitud_media_ha,arroz_exclusión_legal_ha,arroz_no_apta_ha,cod_dane_clean
0,2024-02-07,19809.0,CAUCA,TIMBIQUI,INUNDACION,2468.0,2611.0,629.0,NaN,NaN,...,0.0,0.000000,1645.159805,204451.532475,0.000000,1973.829426,114.632229,1645.159805,202363.070825,19809
1,2024-03-13,27430.0,CHOCO,MEDIO BAUDO,INUNDACION,4342.0,12500.0,4050.0,760.0,7.0,...,0.0,0.000000,4349.765489,133127.915373,0.000000,241.782087,218.994989,4349.765488,132667.138298,27430
2,2024-03-13,27025.0,CHOCO,ALTO BAUDO,INUNDACION,4352.0,15000.0,7500.0,23000.0,35.0,...,0.0,0.000000,23332.980257,183398.918015,0.000000,1020.546290,769.180590,23332.980258,181609.191137,27025
3,2024-04-01,19809.0,CAUCA,TIMBIQUI,INUNDACION,5207.0,3391.0,980.0,633.0,4.0,...,0.0,0.000000,1645.159805,204451.532475,0.000000,1973.829426,114.632229,1645.159805,202363.070825,19809
4,2024-04-05,5147.0,ANTIOQUIA,CAREPA,INUNDACION,5304.0,7500.0,1875.0,270.0,NaN,...,0.0,337.622796,480.919467,20031.367213,21.045376,9715.520773,9627.856537,480.919468,18748.569942,05147


Una vez realizada la unión entre los datos de la UNGRD y la Aptitud de los dos tipos de cultivos y verificar que se hizo correctamente se pasa a la unión de el resultado y la exposición EVA.

###### Union entre el resultado (union entre los datos UNGRD y la Aptitudes del maiz y arroz) y exposicion EVA. 

In [60]:
# Unión entre el resultado + Producción Anual (EVA)
df_master = df_master.merge(
    df_exposicion_eva, 
    left_on=['DIVIPOLA_clean', 'year_clean'], 
    right_on=['cod_dane_clean', 'Año_clean'], 
    how='left'
)

In [61]:
info_data(df_master)

El dataset consta de  (1192, 50)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [62]:
# Aseguramos que la llave en df_master esté limpia antes del cruce
df_master['DIVIPOLA_clean'] = df_master['DIVIPOLA'].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(5)

df_master = df_master.merge(
    df_frontera_muni,
    left_on='DIVIPOLA_clean',
    right_on='cod_dane_clean',
    how='left'
)

In [63]:
# Llenamos con 0 los casos raros de municipios sin frontera agrícola reportada
df_master['frontera_agricola_total_ha'] = df_master['frontera_agricola_total_ha'].fillna(0)

# Limpieza final de llaves
df_master = df_master.drop(columns=['DIVIPOLA_clean', 'cod_dane_clean', 'Año_clean'], errors='ignore')

Una vez realizada la unión de los tres dataset, se pasa a borrarle las columnas sobrantes (Limpiar el dataset)

In [64]:
# Eliminamos las columnas llave duplicadas o temporales para no ensuciar el modelo
columnas_a_borrar = [
    'cod_dane_m', 'Código Dane municipio', 'Año', 
    'DIVIPOLA_clean', 'year_clean', 'cod_dane_clean_x', 'cod_dane_clean_y', 'Año_clean'
]
# Borramos solo las que existan para evitar errores
df_master = df_master.drop(columns=[col for col in columnas_a_borrar if col in df_master.columns])

In [65]:
# Eliminamos las columnas llave duplicadas o temporales para no ensuciar el modelo
columnas_a_borrar = [
    'cod_dane_m', 'Código Dane municipio', 'Año', 
    'DIVIPOLA_clean', 'year_clean', 'cod_dane_clean_x', 'cod_dane_clean_y', 'Año_clean'
]
# Borramos solo las que existan para evitar errores
df_master = df_master.drop(columns=[col for col in columnas_a_borrar if col in df_master.columns])

In [66]:
# Borramos todas las columnas que terminan en '_y' (los duplicados)
columnas_y = [col for col in df_master.columns if str(col).endswith('_y')]
df_master = df_master.drop(columns=columnas_y)

# Le quitamos el sufijo '_x' a las columnas originales para que queden limpias
df_master.columns = df_master.columns.str.replace('_x', '', regex=False)

###### Union entre el dataset ya preparado y las frontera agricola

##### Resultado de la unión

In [67]:
info_data(df_master)

El dataset consta de  (1192, 43)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [68]:
print(f"La base maestra tiene ahora {df_master.shape[1]} columnas.")
print("Muestra de las hectáreas de frontera agrícola por evento:")
display(df_master[['DIVIPOLA', 'MUNICIPIO', 'frontera_agricola_total_ha']].head())

La base maestra tiene ahora 43 columnas.
Muestra de las hectáreas de frontera agrícola por evento:


,DIVIPOLA,MUNICIPIO,frontera_agricola_total_ha
0,19809.0,TIMBIQUI,12917.216999
1,27430.0,MEDIO BAUDO,6319.188190
2,27025.0,ALTO BAUDO,11903.200548
3,19809.0,TIMBIQUI,12917.216999
4,5147.0,CAREPA,32244.545202


In [69]:
datos_nulos(df_master)

,nulos,% nulos,tipo
ACUED.,1041,87.33,float64
VIV.DESTRU.,1008,84.56,float64
VIAS,936,78.52,float64
OTROS,909,76.26,object
HECTAREAS,758,63.59,float64
VALOR.1,754,63.26,float64
support_fngrd_value,754,63.26,float64
maíz_área_cosechada_(ha),562,47.15,float64
arroz_área_cosechada_(ha),562,47.15,float64
arroz_producción_(t),562,47.15,float64


## 2Da parte

Ahora sigue el proceso de dejar una sola base de datos entre los registros de precipitación y UNGRD, además teniendo en cuenta que los eventos ocurren en temporadas o inesperadamente las variables principales para unir sería las fechas, es decir, se quiere que las variables llave sean las fechas, es decir se necesita transformar o sacar variables derivadas de las que ya están ya que por ejemplo: "Como en un solo mes pueden pasar 10 eventos catastroficos de inundaciones o uno se requiere realizar una sumatoria y dejar el resultado como evento de un mes, de ahí deriva un problema de si son varios cual se va areportar, en este caso se reportaría el máximo o el evento que causo mayor daño.".

Lo que se decide vovler a normalizar ambos datasets (Precipitaciones - UNGRD/Cultivos) entiendanse que el primer dataset es precipitaciones que contiene los datos topograficos, la precipitación mensual, entre otras caracteristicas metereologicas, etc. Por otro lado, los datos de UNGRD/eventos contiene los eventos, registros de las hectares y cultivos como de arroz y de maiz.

Antes de comenzar se borran algunas variables para no causar daños o reinicios por la capacidad del computador.

In [70]:
#del df_aptitud_total, df_arroz_final, df_eva_filtrado, df_exposicion_eva, df_maiz_final, df_frontera_muni

### Funciones

In [71]:
# Función para limpiar el texto
def normalize_text(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip().upper()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    x = re.sub(r"\s+", " ", x)
    return x

In [72]:
# Función para limpiar divipola, así creamos una llave fija y segura aparte de las fechas
def clean_divipola(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    s = s.replace(".0", "")
    s = re.sub(r"[^\d]", "", s)
    if s == "":
        return pd.NA
    if len(s) < 5:
        s = s.zfill(5)
    return s

In [73]:
# Función para limpiar las variables numericas, también lo determinamos en miles
def clean_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    s = str(x).strip()
    if s == "":
        return np.nan

    s = s.replace("$", "").replace("COP", "").replace("cop", "")
    s = s.replace("\n", "").replace("\t", "").replace(" ", "")

    # Notación científica
    sci = re.match(r"^-?\d+(\.\d+)?[eE][+-]?\d+$", s)
    if sci:
        try:
            return float(s)
        except:
            return np.nan

    # Un IF para los miles con coma y decimal con punto
    if "," in s and "." in s:
        if s.rfind(".") > s.rfind(","):
            s = s.replace(",", "")
        else:
            s = s.replace(".", "").replace(",", ".")

    # Uns egundo IF para las comas
    elif "," in s and "." not in s:
        s = s.replace(",", "")

    # Un tecer IF para varios puntos
    elif s.count(".") > 1:
        s = s.replace(".", "")

    # Se elimina todo excepto dígitos, signo y punto
    s = re.sub(r"[^0-9\.\-]", "", s)

    try:
        return float(s)
    except:
        return np.nan

In [74]:
# Función para las series largas 
def longest_non_null(series):
    vals = [str(x) for x in series.dropna() if str(x).strip() != ""]
    if not vals:
        return pd.NA
    return max(vals, key=len)

In [75]:
def first_non_null(series):
    vals = series.dropna()
    return vals.iloc[0] if len(vals) else pd.NA

In [76]:
def mean_positive_or_zero(series):
    s = pd.to_numeric(series, errors="coerce")
    s = s[s > 0]
    return s.mean() if not s.empty else 0.0

In [77]:
def sum_or_zero(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.fillna(0).sum()

In [78]:
def max_or_zero(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.max() if s.notna().any() else 0.0

## Preparación del dataset

In [79]:
# Copiamos el dataset para no embarrarla
df = df_master.copy()

In [80]:
# Normalizamos 
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")
df["DIVIPOLA"] = df["DIVIPOLA"].apply(clean_divipola)

df["DEPARTAMENTO"] = df["DEPARTAMENTO"].apply(normalize_text)
df["MUNICIPIO"] = df["MUNICIPIO"].apply(normalize_text)
df["EVENTO"] = df["EVENTO"].apply(normalize_text)

In [81]:
# Recalcular tiempo desde FECHA
df["year"] = df["FECHA"].dt.year
df["month"] = df["FECHA"].dt.month
df["ym"] = df["FECHA"].dt.to_period("M")


In [82]:
# Limpiamos los numericos
# Limpiar numéricos
numeric_cols = [
    "CODIGO EMERGENCIA", "PERSONAS", "FAMILIAS", "HECTAREAS",
    "VIV.DESTRU.", "VIV.AVER.", "VIAS", "ACUED.",
    "VALOR.1", "event_id_clean",
    "housing_damage_proxy", "infra_damage_proxy", "support_fngrd_value"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

In [83]:
# Creamos los proxies
if "housing_damage_proxy" not in df.columns:
    df["housing_damage_proxy"] = (
        df["VIV.DESTRU."].fillna(0) + df["VIV.AVER."].fillna(0)
    )

if "infra_damage_proxy" not in df.columns:
    df["infra_damage_proxy"] = (
        df["VIAS"].fillna(0) + df["ACUED."].fillna(0)
    )

if "support_fngrd_value" not in df.columns:
    df["support_fngrd_value"] = df["VALOR.1"]

In [84]:
# Creamos las variables banderas
if "missing_divipola_flag" not in df.columns:
    df["missing_divipola_flag"] = df["DIVIPOLA"].isna().astype(int)

if "hectareas_reported_flag" not in df.columns:
    df["hectareas_reported_flag"] = df["HECTAREAS"].gt(0).fillna(False).astype(int)


In [85]:
info_data(df)

El dataset consta de  (1192, 43)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [86]:
# Creamos la variable flood_crop_text_flag
if "flood_crop_text_flag" not in df.columns:
    texto = (
        df["OTROS"].fillna("").astype(str) + " " +
        df["COMENTARIOS"].fillna("").astype(str)
    ).str.upper()

    patrones = [
        r"\bPERDID[AO]S?\s+DE\s+CULTIVOS?\b",
        r"\bDANOS?\s+EN\s+CULTIVOS?\b",
        r"\bCULTIVOS?\s+AFECTADOS?\b",
        r"\bCULTIVOS?\s+DE\s+PANCOGER\b",
        r"\bPLATANO\b", r"\bBANANO\b", r"\bYUCA\b", r"\bMAIZ\b",
        r"\bARROZ\b", r"\bCACAO\b", r"\bPAPAYA\b", r"\bCITRICOS\b"
    ]
    regex = "|".join(patrones)
    df["flood_crop_text_flag"] = texto.str.contains(regex, regex=True, na=False).astype(int)

if "flood_crop_quantified_flag" not in df.columns:
    df["flood_crop_quantified_flag"] = df["HECTAREAS"].gt(0).fillna(False).astype(int)

if "flood_crop_any_flag" not in df.columns:
    df["flood_crop_any_flag"] = (
        (df["flood_crop_text_flag"] == 1) |
        (df["flood_crop_quantified_flag"] == 1)
    ).astype(int)

In [87]:
# Creamos una bandera util para control de calidad
df["crop_text_only_flag"] = (
    (df["flood_crop_text_flag"] == 1) &
    (df["hectareas_reported_flag"] == 0)
).astype(int)

In [88]:
info_data(df)

El dataset consta de  (1192, 44)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 44 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

Una vez realizado el primer filtro o limpieza de las fechas y algunas variables de cultilvos, se procede a normalizar/limpiar los códigos

In [89]:
# Normalizamos los códigos o ids
df["event_id_clean_str"] = df["event_id_clean"].apply(
    lambda x: str(int(x)) if pd.notna(x) else pd.NA
)
df["codigo_emergencia_str"] = df["CODIGO EMERGENCIA"].apply(
    lambda x: str(int(x)) if pd.notna(x) else pd.NA
)

In [90]:
# Se rellena un registro en el caso que no haya código DIVIPOLA
plan_b = (
    df["DIVIPOLA"].fillna("00000").astype(str) + "_" +
    df["FECHA"].dt.strftime("%Y%m%d").fillna("00000000") + "_" +
    df["EVENTO"].fillna("SIN_EVENTO").astype(str)
)

df["event_group_key"] = np.where(
    df["event_id_clean_str"].notna(),
    df["DIVIPOLA"].fillna("00000").astype(str) + "_EID_" + df["event_id_clean_str"].astype(str),
    np.where(
        df["codigo_emergencia_str"].notna(),
        df["DIVIPOLA"].fillna("00000").astype(str) + "_CEM_" + df["codigo_emergencia_str"].astype(str),
        plan_b
    )
)

In [91]:
info_data(df)

El dataset consta de  (1192, 47)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 1192 entries, 0 to 1191
Data columns (total 47 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   FECHA                       1192 non-null   datetime64[us]
 1   DIVIPOLA                    1164 non-null   str           
 2   DEPARTAMENTO                1192 non-null   str           
 3   MUNICIPIO                   1192 non-null   str           
 4   EVENTO                      1192 non-null   str           
 5   CODIGO EMERGENCIA           958 non-null    float64       
 6   PERSONAS                    939 non-null    float64       
 7   FAMILIAS                    986 non-null    float64       
 8   HECTAREAS                   434 non-null    float64       
 9   VIV.DESTRU.                 184 non-null    float64       
 10  VIV.AVER.                   755 non-null    float64 

In [92]:
info_data(plan_b)

El dataset consta de  (1192,)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.Series'>
RangeIndex: 1192 entries, 0 to 1191
Series name: None
Non-Null Count  Dtype
--------------  -----
1192 non-null   str  
dtypes: str(1)
memory usage: 38.5 KB


Una vez ya realizado la limpieza de los códigos, se pasa a la unión o consolidación del dataset

Una vez reaoganizado y con las variables necesaria para el modelo procedemos a agregar/unir el dataset de precipitaciones con el de la UNGRD

In [93]:
df_precipitacion = pd.read_csv('../Data_preparada/dataset_ml_municipal_monthly.csv')

In [94]:
info_data(df_precipitacion)

El dataset consta de  (199716, 21)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 199716 entries, 0 to 199715
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   system:index          199716 non-null  str    
 1   MpCodigo              199716 non-null  int64  
 2   month                 199716 non-null  int64  
 3   precip_mm_month_mean  199360 non-null  float64
 4   year                  199716 non-null  int64  
 5   ym                    199716 non-null  str    
 6   .geo                  199716 non-null  str    
 7   MpNombre              199716 non-null  str    
 8   Depto_Codigo          199716 non-null  int64  
 9   MpAltitud             199716 non-null  int64  
 10  area_geom_km2         199716 non-null  float64
 11  len_km                199716 non-null  float64
 12  drenaje_km_por_km2    199716 non-null  float64
 13  dist_dren_m           199716 non-n

In [95]:
# Compiamos para no tener que cargar nuevamente el script en caso de un error
panel_base = df_precipitacion.copy()
panel_climatico = df_precipitacion.copy()

Unimos los datasets para consolidad el dataset final

Ya quedó el dataset final de dando con 199716 registros/filas y 43 columnas, además los eventos agricolas están bajos con mean	0.005473, 0.003720 de flood_any_it	flood_crop_any_it respectivamente, lo que indica un desbalance.

Renombramos todas las variables al español para luego derivar dos bases de la final.

### Re-organizamos

Se reorganiza debido a la combinación erronea que se hizo anteriormente al mezclar todo en un solo dataset.

In [96]:
def primero_no_nulo(serie):
    s = serie.dropna()
    return s.iloc[0] if len(s) else pd.NA

def maximo_o_cero(serie):
    s = pd.to_numeric(serie, errors="coerce")
    return s.max() if s.notna().any() else 0

def suma_o_cero(serie):
    s = pd.to_numeric(serie, errors="coerce")
    return s.fillna(0).sum()

def texto_mas_largo(serie):
    vals = [str(x) for x in serie.dropna() if str(x).strip() != ""]
    return max(vals, key=len) if vals else pd.NA

Se arma el catalogo municipal desde la base climatica

In [97]:
catalogo_municipios = (
    panel_base[["MpCodigo", "MpNombre", "Depto_Codigo"]]
    .drop_duplicates()
    .copy()
)

catalogo_municipios["codigo_municipio"] = catalogo_municipios["MpCodigo"].astype(str).str.zfill(5)
catalogo_municipios["municipio"] = catalogo_municipios["MpNombre"].astype(str).str.upper().str.strip()
catalogo_municipios["codigo_departamento"] = catalogo_municipios["Depto_Codigo"].astype(str).str.zfill(2)

catalogo_municipios = catalogo_municipios[["codigo_municipio", "municipio", "codigo_departamento"]].drop_duplicates()

Se crea una tabla UNGRD limpia solo con variables evento

In [98]:
base_eventos = df_ungrd.copy()

base_eventos = base_eventos.rename(columns={
    "DIVIPOLA": "codigo_municipio",
    "DEPARTAMENTO": "departamento",
    "MUNICIPIO": "municipio",
    "EVENTO": "evento",
    "FECHA": "fecha_evento",
    "CODIGO EMERGENCIA": "codigo_emergencia",
    "PERSONAS": "personas_afectadas",
    "FAMILIAS": "familias_afectadas",
    "HECTAREAS": "hectareas_afectadas",
    "VIV.DESTRU.": "viviendas_destruidas",
    "VIV.AVER.": "viviendas_averiadas",
    "VIAS": "vias_afectadas",
    "ACUED.": "acueductos_afectados",
    "OTROS": "otros_danios",
    "COMENTARIOS": "comentarios",
    "VALOR.1": "valor_apoyo_fngrd"
})

base_eventos["codigo_municipio"] = base_eventos["codigo_municipio"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
base_eventos["municipio"] = base_eventos["municipio"].astype(str).str.upper().str.strip()
base_eventos["departamento"] = base_eventos["departamento"].astype(str).str.upper().str.strip()

base_eventos["anio"] = pd.to_datetime(base_eventos["fecha_evento"], errors="coerce").dt.year
base_eventos["mes"] = pd.to_datetime(base_eventos["fecha_evento"], errors="coerce").dt.month
base_eventos["periodo"] = pd.to_datetime(base_eventos["fecha_evento"], errors="coerce").dt.to_period("M")

Se imputa el codigo DIVIPOLA

In [99]:
# Si no tienes departamento homologado, este paso habrá que afinarlo
mapa_municipio = (
    panel_base[["MpCodigo", "MpNombre"]]
    .drop_duplicates()
    .assign(
        codigo_municipio=lambda x: x["MpCodigo"].astype(str).str.zfill(5),
        municipio=lambda x: x["MpNombre"].astype(str).str.upper().str.strip()
    )[["codigo_municipio", "municipio"]]
)

faltantes = base_eventos["codigo_municipio"].isna() | (base_eventos["codigo_municipio"] == "0".zfill(5))

base_eventos = base_eventos.merge(
    mapa_municipio.rename(columns={"codigo_municipio": "codigo_municipio_catalogo"}),
    on="municipio",
    how="left"
)

base_eventos["codigo_municipio"] = base_eventos["codigo_municipio"].fillna(base_eventos["codigo_municipio_catalogo"])
base_eventos = base_eventos.drop(columns=["codigo_municipio_catalogo"])

Se separa UPRA en dos tablas, Upra estatico la cual posee aptitud maiz, aptitud arroz, restricciones y frontera agricola.

In [100]:
columnas_upra_estatica = [
    "DIVIPOLA", 
    "maiz_aptitud_alta_ha", "maiz_aptitud_baja_ha", "maiz_aptitud_media_ha",
    "maiz_exclusión_legal_ha", "maiz_no_apta_ha",
    "arroz_aptitud_alta_ha", "arroz_aptitud_baja_ha", "arroz_aptitud_media_ha",
    "arroz_exclusión_legal_ha", "arroz_no_apta_ha",
    "frontera_agricola_total_ha"
]

upra_estatica = df_master[columnas_upra_estatica].copy()
upra_estatica = upra_estatica.rename(columns={"DIVIPOLA": "codigo_municipio"})
upra_estatica["codigo_municipio"] = upra_estatica["codigo_municipio"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
upra_estatica = upra_estatica.drop_duplicates(subset=["codigo_municipio"])

Upra anual Una fila por municipio año

In [101]:
columnas_upra_anual = [
    "DIVIPOLA", "year",
    "arroz_producción_(t)", "maíz_producción_(t)",
    "arroz_área_cosechada_(ha)", "maíz_área_cosechada_(ha)"
]

upra_anual = df_master[columnas_upra_anual].copy()
upra_anual = upra_anual.rename(columns={
    "DIVIPOLA": "codigo_municipio",
    "year": "anio",
    "arroz_producción_(t)": "arroz_produccion_t",
    "maíz_producción_(t)": "maiz_produccion_t",
    "arroz_área_cosechada_(ha)": "arroz_area_cosechada_ha",
    "maíz_área_cosechada_(ha)": "maiz_area_cosechada_ha"
})

upra_anual["codigo_municipio"] = upra_anual["codigo_municipio"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
upra_anual["anio"] = pd.to_numeric(upra_anual["anio"], errors="coerce")

upra_anual = upra_anual.drop_duplicates(subset=["codigo_municipio", "anio"])

Se consolida los eventos UNGRD sin tener en cuenta UPRA

In [102]:
base_eventos["id_evento_limpio"] = base_eventos["event_id_clean"].apply(lambda x: str(int(x)) if pd.notna(x) else pd.NA)
base_eventos["codigo_emergencia_limpio"] = base_eventos["codigo_emergencia"].apply(lambda x: str(int(x)) if pd.notna(x) else pd.NA)

llave_respaldo = (
    base_eventos["codigo_municipio"].fillna("00000").astype(str) + "_" +
    pd.to_datetime(base_eventos["fecha_evento"]).dt.strftime("%Y%m%d").fillna("00000000") + "_" +
    base_eventos["evento"].fillna("SIN_EVENTO").astype(str)
)

base_eventos["llave_grupo_evento"] = np.where(
    base_eventos["id_evento_limpio"].notna(),
    base_eventos["codigo_municipio"].fillna("00000").astype(str) + "_EID_" + base_eventos["id_evento_limpio"].astype(str),
    np.where(
        base_eventos["codigo_emergencia_limpio"].notna(),
        base_eventos["codigo_municipio"].fillna("00000").astype(str) + "_CEM_" + base_eventos["codigo_emergencia_limpio"].astype(str),
        llave_respaldo
    )
)

Se consolidan los eventos

In [103]:
eventos_consolidados = (
    base_eventos
    .groupby("llave_grupo_evento", as_index=False)
    .agg({
        "codigo_municipio": primero_no_nulo,
        "departamento": primero_no_nulo,
        "municipio": primero_no_nulo,
        "evento": primero_no_nulo,
        "fecha_evento": "min",
        "personas_afectadas": maximo_o_cero,
        "familias_afectadas": maximo_o_cero,
        "hectareas_afectadas": maximo_o_cero,
        "viviendas_destruidas": maximo_o_cero,
        "viviendas_averiadas": maximo_o_cero,
        "vias_afectadas": maximo_o_cero,
        "acueductos_afectados": maximo_o_cero,
        "otros_danios": texto_mas_largo,
        "comentarios": texto_mas_largo,
        "valor_apoyo_fngrd": maximo_o_cero,
        "flood_crop_text_flag": "max",
        "hectareas_reported_flag": "max",
        "flood_crop_quantified_flag": "max",
        "flood_crop_any_flag": "max",
        "missing_divipola_flag": "max"
    })
)

eventos_consolidados["anio"] = pd.to_datetime(eventos_consolidados["fecha_evento"]).dt.year
eventos_consolidados["mes"] = pd.to_datetime(eventos_consolidados["fecha_evento"]).dt.month
eventos_consolidados["periodo"] = pd.to_datetime(eventos_consolidados["fecha_evento"]).dt.to_period("M")

eventos_consolidados["proxy_danio_vivienda"] = (
    eventos_consolidados["viviendas_destruidas"].fillna(0) +
    eventos_consolidados["viviendas_averiadas"].fillna(0)
)

eventos_consolidados["proxy_danio_infraestructura"] = (
    eventos_consolidados["vias_afectadas"].fillna(0) +
    eventos_consolidados["acueductos_afectados"].fillna(0)
)

Se agrega municipio mes para generar llave

In [104]:
ungrd_municipio_mes = (
    eventos_consolidados
    .groupby(["codigo_municipio", "anio", "mes", "periodo"], as_index=False)
    .agg(
        hubo_inundacion_mes=("llave_grupo_evento", lambda x: 1),
        hubo_inundacion_agricola_mes=("flood_crop_any_flag", "max"),
        numero_eventos_inundacion_mes=("llave_grupo_evento", "nunique"),
        numero_eventos_agricolas_mes=("flood_crop_any_flag", "sum"),
        hectareas_afectadas_suma_mes=("hectareas_afectadas", suma_o_cero),
        hectareas_afectadas_max_mes=("hectareas_afectadas", maximo_o_cero),
        personas_afectadas_suma_mes=("personas_afectadas", suma_o_cero),
        familias_afectadas_suma_mes=("familias_afectadas", suma_o_cero),
        danio_vivienda_suma_mes=("proxy_danio_vivienda", suma_o_cero),
        danio_infraestructura_suma_mes=("proxy_danio_infraestructura", suma_o_cero),
        apoyo_fngrd_suma_mes=("valor_apoyo_fngrd", suma_o_cero),
        numero_eventos_con_hectareas_mes=("hectareas_reported_flag", "sum"),
        numero_eventos_sin_divipola_mes=("missing_divipola_flag", "sum")
    )
)

ungrd_municipio_mes["indicador_reporte_hectareas_mes"] = (
    ungrd_municipio_mes["numero_eventos_con_hectareas_mes"] > 0
).astype("int8")

Se empieza a construir el panel final

In [105]:
#panel_climatico = data_complete.copy()

panel_climatico = panel_climatico.rename(columns={
    "MpCodigo": "codigo_municipio",
    "month": "mes",
    "year": "anio",
    "ym": "periodo",
    "MpNombre": "nombre_municipio",
    "Depto_Codigo": "codigo_departamento",
    "MpAltitud": "altitud_municipio_m",
    "area_geom_km2": "area_municipio_km2",
    "len_km": "longitud_drenajes_km",
    "drenaje_km_por_km2": "densidad_drenaje_km_km2",
    "dist_dren_m": "distancia_drenaje_m",
    "wet_pct_area": "proporcion_humedales",
    "elev_mean_m": "elevacion_media_m",
    "slope_mean_deg": "pendiente_media_grados",
    "precip_mm_month_mean": "precipitacion_media_mensual_mm",
    "precip_3m_sum": "precipitacion_acumulada_3m_mm",
    "precip_3m_mean": "precipitacion_media_3m_mm",
    "p95_mun": "percentil_95_precipitacion_municipal_mm",
    "is_extreme_month": "indicador_mes_extremo"
})

panel_climatico["codigo_municipio"] = panel_climatico["codigo_municipio"].astype(str).str.zfill(5)
panel_climatico["anio"] = pd.to_numeric(panel_climatico["anio"], errors="coerce").astype(int)
panel_climatico["mes"] = pd.to_numeric(panel_climatico["mes"], errors="coerce").astype(int)
panel_climatico["periodo"] = pd.PeriodIndex(panel_climatico["periodo"], freq="M")

Unimos los datasets

In [106]:
panel_modelo = (
    panel_climatico
    .merge(ungrd_municipio_mes, on=["codigo_municipio", "anio", "mes"], how="left")
    .merge(upra_anual, on=["codigo_municipio", "anio"], how="left")
    .merge(upra_estatica, on=["codigo_municipio"], how="left")
)

In [107]:
columnas_evento_cero = [
    "hubo_inundacion_mes",
    "hubo_inundacion_agricola_mes",
    "numero_eventos_inundacion_mes",
    "numero_eventos_agricolas_mes",
    "hectareas_afectadas_suma_mes",
    "hectareas_afectadas_max_mes",
    "personas_afectadas_suma_mes",
    "familias_afectadas_suma_mes",
    "danio_vivienda_suma_mes",
    "danio_infraestructura_suma_mes",
    "apoyo_fngrd_suma_mes",
    "numero_eventos_con_hectareas_mes",
    "numero_eventos_agricolas_texto_sin_hectareas_mes",
    "numero_eventos_sin_divipola_mes",
    "indicador_reporte_hectareas_mes"
]

for col in columnas_evento_cero:
    if col in panel_modelo.columns:
        panel_modelo[col] = panel_modelo[col].fillna(0)

### Resultados v3

Estos son los resultados despues de voolver a consolidar los datasets despues de error al unirlos anteriormente.

In [108]:
columnas_upra = [
    "maiz_aptitud_alta_ha",
    "arroz_aptitud_alta_ha",
    "frontera_agricola_total_ha",
    "arroz_produccion_t",
    "maiz_produccion_t",
    "arroz_area_cosechada_ha",
    "maiz_area_cosechada_ha"
]

panel_modelo[columnas_upra].info()

<class 'pandas.DataFrame'>
RangeIndex: 199716 entries, 0 to 199715
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   maiz_aptitud_alta_ha        81524 non-null  float64
 1   arroz_aptitud_alta_ha       81524 non-null  float64
 2   frontera_agricola_total_ha  81524 non-null  float64
 3   arroz_produccion_t          5637 non-null   float64
 4   maiz_produccion_t           5637 non-null   float64
 5   arroz_area_cosechada_ha     5637 non-null   float64
 6   maiz_area_cosechada_ha      5637 non-null   float64
dtypes: float64(7)
memory usage: 10.7 MB


In [109]:
llaves_ungrd_sin_clima = ungrd_municipio_mes.merge(
    panel_climatico[["codigo_municipio", "anio", "mes"]].drop_duplicates(),
    on=["codigo_municipio", "anio", "mes"],
    how="left",
    indicator=True
)

llaves_ungrd_sin_clima = llaves_ungrd_sin_clima[llaves_ungrd_sin_clima["_merge"] == "left_only"]
print(llaves_ungrd_sin_clima.shape)
llaves_ungrd_sin_clima.head()

(5, 19)


,codigo_municipio,anio,mes,periodo,hubo_inundacion_mes,hubo_inundacion_agricola_mes,numero_eventos_inundacion_mes,numero_eventos_agricolas_mes,hectareas_afectadas_suma_mes,hectareas_afectadas_max_mes,personas_afectadas_suma_mes,familias_afectadas_suma_mes,danio_vivienda_suma_mes,danio_infraestructura_suma_mes,apoyo_fngrd_suma_mes,numero_eventos_con_hectareas_mes,numero_eventos_sin_divipola_mes,indicador_reporte_hectareas_mes,_merge
22,05172,2014,12,2014-12,1,1,1,1,0.0,0.0,1500.0,300.0,0.0,0.0,0.0,0,0,0,left_only
256,19212,2014,12,2014-12,1,1,1,1,0.0,0.0,615.0,100.0,100.0,6.0,0.0,0,0,0,left_only
565,27430,2014,12,2014-12,1,0,1,0,0.0,0.0,8950.0,1790.0,0.0,0.0,0.0,0,0,0,left_only
921,76109,2019,12,2019-12,1,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,left_only
926,76275,2014,12,2014-12,1,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,left_only


In [110]:
panel_modelo.groupby("anio")[["arroz_area_cosechada_ha", "maiz_area_cosechada_ha"]].apply(lambda x: x.notna().mean())

,arroz_area_cosechada_ha,maiz_area_cosechada_ha
anio,,
2010,0.000000,0.000000
2011,0.000000,0.000000
2012,0.000000,0.000000
2013,0.000000,0.000000
2014,0.000000,0.000000
2015,0.000000,0.000000
2016,0.000000,0.000000
2017,0.000000,0.000000
2018,0.000000,0.000000


Se puede observar que lso datos de UPRA estatica tiene 81.524 no nulos, lo que equivale al 40.8% del panel, se extraerá directamente del dataset, para ver si mejora este resultado.
Por otro lado, con los datos de EVA se muestran con alrededor de 5.637 no nulos y desde el 2019 que esto ya se sabía dado a que solo ofrecen hasta esa información.

Se decide arreglar el tema extrayendo directamente desde los datasets limpios.


## Re-organizamos v4

Se crea el catalogo base

In [111]:
catalogo_municipios = (
    panel_climatico[["codigo_municipio", "nombre_municipio", "codigo_departamento"]]
    .drop_duplicates()
    .copy()
)

catalogo_municipios["codigo_municipio"] = (
    catalogo_municipios["codigo_municipio"].astype(str).str.zfill(5)
)

catalogo_anios = pd.DataFrame({
    "anio": sorted(panel_climatico["anio"].dropna().astype(int).unique())
})

catalogo_municipio_anio = (
    catalogo_municipios.assign(llave=1)
    .merge(catalogo_anios.assign(llave=1), on="llave", how="outer")
    .drop(columns="llave")
)

Se reconstruye upra_estatica desde la fuente orignal

In [112]:
# Aptitud total
df_aptitud_total = df_aptitud_total.copy()
df_aptitud_total["codigo_municipio"] = (
    df_aptitud_total["cod_dane_m"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
)

# Frontera agricola
df_frontera_muni = df_frontera_muni.copy()
df_frontera_muni["codigo_municipio"] = (
    df_frontera_muni["cod_dane_clean"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
)

# Restricciones
df_exclusiones_muni = df_exclusiones_muni.copy()
df_exclusiones_muni["codigo_municipio"] = (
    df_exclusiones_muni["cod_dane_mpio"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
)

# Merge de UPRA estatica desde las fuentes originales
upra_estatica_completa = (
    catalogo_municipios[["codigo_municipio"]]
    .merge(
        df_aptitud_total[[
            "codigo_municipio",
            "maiz_aptitud_alta_ha", "maiz_aptitud_baja_ha", "maiz_aptitud_media_ha",
            "maiz_exclusión_legal_ha", "maiz_no_apta_ha",
            "arroz_aptitud_alta_ha", "arroz_aptitud_baja_ha", "arroz_aptitud_media_ha",
            "arroz_exclusión_legal_ha", "arroz_no_apta_ha"
        ]].drop_duplicates("codigo_municipio"),
        on="codigo_municipio",
        how="left"
    )
    .merge(
        df_frontera_muni[["codigo_municipio", "frontera_agricola_total_ha"]].drop_duplicates("codigo_municipio"),
        on="codigo_municipio",
        how="left"
    )
    .merge(
        df_exclusiones_muni[["codigo_municipio", "area_excluida_legal_ha"]].drop_duplicates("codigo_municipio"),
        on="codigo_municipio",
        how="left"
    )
)

columnas_estaticas = [
    "maiz_aptitud_alta_ha", "maiz_aptitud_baja_ha", "maiz_aptitud_media_ha",
    "maiz_exclusión_legal_ha", "maiz_no_apta_ha",
    "arroz_aptitud_alta_ha", "arroz_aptitud_baja_ha", "arroz_aptitud_media_ha",
    "arroz_exclusión_legal_ha", "arroz_no_apta_ha",
    "frontera_agricola_total_ha", "area_excluida_legal_ha"
]

for col in columnas_estaticas:
    if col in upra_estatica_completa.columns:
        upra_estatica_completa[col] = upra_estatica_completa[col].fillna(0)

Se reconstruye upra_anual desde EVA orignal

In [113]:
df_exposicion_eva = df_exposicion_eva.copy()

df_exposicion_eva["codigo_municipio"] = (
    df_exposicion_eva["Código Dane municipio"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(5)
)
df_exposicion_eva["anio"] = pd.to_numeric(df_exposicion_eva["Año"], errors="coerce").astype("Int64")

upra_anual_origen = df_exposicion_eva.rename(columns={
    "arroz_producción_(t)": "arroz_produccion_t",
    "maíz_producción_(t)": "maiz_produccion_t",
    "arroz_área_cosechada_(ha)": "arroz_area_cosechada_ha",
    "maíz_área_cosechada_(ha)": "maiz_area_cosechada_ha"
})

upra_anual_origen = upra_anual_origen[[
    "codigo_municipio", "anio",
    "arroz_produccion_t", "maiz_produccion_t",
    "arroz_area_cosechada_ha", "maiz_area_cosechada_ha"
]].drop_duplicates(subset=["codigo_municipio", "anio"])

upra_anual_completa = (
    catalogo_municipio_anio[["codigo_municipio", "anio"]]
    .merge(upra_anual_origen, on=["codigo_municipio", "anio"], how="left")
)

columnas_anuales = [
    "arroz_produccion_t", "maiz_produccion_t",
    "arroz_area_cosechada_ha", "maiz_area_cosechada_ha"
]

for col in columnas_anuales:
    if col in upra_anual_completa.columns:
        upra_anual_completa[col] = upra_anual_completa[col].fillna(0)

Se vuelve a construir el panel final 

In [114]:
panel_modelo_final = (
    panel_climatico
    .merge(ungrd_municipio_mes, on=["codigo_municipio", "anio", "mes"], how="left")
    .merge(upra_anual_completa, on=["codigo_municipio", "anio"], how="left")
    .merge(upra_estatica_completa, on=["codigo_municipio"], how="left")
)

columnas_evento_cero = [
    "hubo_inundacion_mes",
    "hubo_inundacion_agricola_mes",
    "numero_eventos_inundacion_mes",
    "numero_eventos_agricolas_mes",
    "hectareas_afectadas_suma_mes",
    "hectareas_afectadas_max_mes",
    "personas_afectadas_suma_mes",
    "familias_afectadas_suma_mes",
    "danio_vivienda_suma_mes",
    "danio_infraestructura_suma_mes",
    "apoyo_fngrd_suma_mes",
    "numero_eventos_con_hectareas_mes",
    "numero_eventos_sin_divipola_mes",
    "indicador_reporte_hectareas_mes"
]

for col in columnas_evento_cero:
    if col in panel_modelo_final.columns:
        panel_modelo_final[col] = panel_modelo_final[col].fillna(0)

#### Resultado v4

In [115]:
# Cobertura estatica UPRA
panel_modelo_final[[
    "maiz_aptitud_alta_ha",
    "arroz_aptitud_alta_ha",
    "frontera_agricola_total_ha",
    "area_excluida_legal_ha"
]].notna().mean()

maiz_aptitud_alta_ha          1.0
arroz_aptitud_alta_ha         1.0
frontera_agricola_total_ha    1.0
area_excluida_legal_ha        1.0
dtype: float64

In [116]:
# Cobertura anual EVA
panel_modelo_final.groupby("anio")[[
    "arroz_area_cosechada_ha",
    "maiz_area_cosechada_ha"
]].apply(lambda x: x.notna().mean())

,arroz_area_cosechada_ha,maiz_area_cosechada_ha
anio,,
2010,1.0,1.0
2011,1.0,1.0
2012,1.0,1.0
2013,1.0,1.0
2014,1.0,1.0
2015,1.0,1.0
2016,1.0,1.0
2017,1.0,1.0
2018,1.0,1.0


Se crean banderas debido a que el dataset EVA solo posee Datos del 2019 en adelante este confundiría al modelo interpretando posiblemente que no hay cultivos donde si los hay, entonces se puede interpretar de otra manera como "sin dato de EVA, no necesariamente sin cultivo"

In [117]:
panel_modelo_final["indicador_eva_disponible"] = (
    panel_modelo_final["anio"] >= 2019
).astype("int8")

panel_modelo_final["indicador_exposicion_eva_reportada"] = (
    (panel_modelo_final["arroz_area_cosechada_ha"] > 0) |
    (panel_modelo_final["maiz_area_cosechada_ha"] > 0)
).astype("int8")

In [118]:
info_data(panel_modelo_final)

El dataset consta de  (199716, 54)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 199716 entries, 0 to 199715
Data columns (total 54 columns):
 #   Column                                   Non-Null Count   Dtype    
---  ------                                   --------------   -----    
 0   system:index                             199716 non-null  str      
 1   codigo_municipio                         199716 non-null  str      
 2   mes                                      199716 non-null  int64    
 3   precipitacion_media_mensual_mm           199360 non-null  float64  
 4   anio                                     199716 non-null  int64    
 5   periodo_x                                199716 non-null  period[M]
 6   .geo                                     199716 non-null  str      
 7   nombre_municipio                         199716 non-null  str      
 8   codigo_departamento                      199716 non-null  int64    
 9   altitud_mun

### Limpieza final

Como se puede observar en la informaciónd e las variables hay ciertas cosas que limpiar y arreglar, primero eliminar un periodo, elimamos las variables geoespaciales ya que no se neceitan para el modelo. Se renombran algunas variables para evitar problemas con el modelo.

In [119]:
# Eliminamos un periodo
panel_modelo_final = panel_modelo_final.rename(columns={"periodo_x": "periodo"})
if "periodo_y" in panel_modelo_final.columns:
    panel_modelo_final = panel_modelo_final.drop(columns=["periodo_y"])

In [120]:
# Eliminamos las variables geoespaciales
columnas_tecnicas = [c for c in ["system:index", ".geo"] if c in panel_modelo_final.columns]
panel_modelo_final = panel_modelo_final.drop(columns=columnas_tecnicas)

In [121]:
# Limpiamos nos nombres de las variables/columnas
panel_modelo_final.columns = [normalizar_nombre_columna(c) for c in panel_modelo_final.columns]

In [122]:
# Se convierten variables de evento reales a enteros
columnas_binarias = [
    "hubo_inundacion_mes",
    "hubo_inundacion_agricola_mes",
    "indicador_reporte_hectareas_mes",
    "indicador_eva_disponible",
    "indicador_exposicion_eva_reportada"
]

for col in columnas_binarias:
    if col in panel_modelo_final.columns:
        panel_modelo_final[col] = panel_modelo_final[col].astype("int8")

columnas_conteo = [
    "numero_eventos_inundacion_mes",
    "numero_eventos_agricolas_mes",
    "numero_eventos_con_hectareas_mes",
    "numero_eventos_sin_divipola_mes"
]

for col in columnas_conteo:
    if col in panel_modelo_final.columns:
        panel_modelo_final[col] = panel_modelo_final[col].astype("int16")

Por otro lado, se aprovecha la oportunidad de crear variables que pueden servir en el modelado

In [123]:
panel_modelo_final["area_municipio_ha"] = panel_modelo_final["area_municipio_km2"] * 100

panel_modelo_final["proporcion_frontera_agricola"] = np.where(
    panel_modelo_final["area_municipio_ha"] > 0,
    panel_modelo_final["frontera_agricola_total_ha"] / panel_modelo_final["area_municipio_ha"],
    np.nan
)

panel_modelo_final["area_cosechada_total_ha"] = (
    panel_modelo_final["arroz_area_cosechada_ha"] +
    panel_modelo_final["maiz_area_cosechada_ha"]
)

panel_modelo_final["produccion_total_t"] = (
    panel_modelo_final["arroz_produccion_t"] +
    panel_modelo_final["maiz_produccion_t"]
)

panel_modelo_final["proporcion_maiz_aptitud_alta"] = np.where(
    panel_modelo_final["frontera_agricola_total_ha"] > 0,
    panel_modelo_final["maiz_aptitud_alta_ha"] / panel_modelo_final["frontera_agricola_total_ha"],
    np.nan
)

panel_modelo_final["proporcion_arroz_aptitud_alta"] = np.where(
    panel_modelo_final["frontera_agricola_total_ha"] > 0,
    panel_modelo_final["arroz_aptitud_alta_ha"] / panel_modelo_final["frontera_agricola_total_ha"],
    np.nan
)

panel_modelo_final["proporcion_maiz_area_cosechada"] = np.where(
    panel_modelo_final["frontera_agricola_total_ha"] > 0,
    panel_modelo_final["maiz_area_cosechada_ha"] / panel_modelo_final["frontera_agricola_total_ha"],
    np.nan
)

panel_modelo_final["proporcion_arroz_area_cosechada"] = np.where(
    panel_modelo_final["frontera_agricola_total_ha"] > 0,
    panel_modelo_final["arroz_area_cosechada_ha"] / panel_modelo_final["frontera_agricola_total_ha"],
    np.nan
)

panel_modelo_final["tasa_afectacion_agricola"] = np.where(
    panel_modelo_final["area_cosechada_total_ha"] > 0,
    panel_modelo_final["hectareas_afectadas_suma_mes"] / panel_modelo_final["area_cosechada_total_ha"],
    np.nan
)

panel_modelo_final["tasa_afectacion_agricola"] = panel_modelo_final["tasa_afectacion_agricola"].clip(upper=1)

Ya que EVA solo está estabelcida o tenemos registro desde el 2019 está podría alterar el modelo, por lo que es necesario considerar poder realizar varios modelos para ver su comportamiento con el conjunto de datos.

In [124]:
# Indicador/bandera para asegurar que el dat9o está observando el registro o no
panel_modelo_final["indicador_eva_observada_fuente"] = np.where(
    panel_modelo_final["anio"].between(2019, 2024),
    1,
    0
).astype("int8")

Extraemos el modelo con EVA para poder compararlos sin mezclarlos.

In [125]:
panel_modelo_eva = panel_modelo_final[panel_modelo_final["anio"] >= 2019].copy()

Se crea un bandera para tener más consistencia con la variable tasa_afectación_agricola, esto debido aque posee un total de 72.482 nulos, pero podría significar que no había exposición agricola

In [126]:
panel_modelo_final["indicador_exposicion_agricola_positiva"] = (
    panel_modelo_final["area_cosechada_total_ha"] > 0
).astype("int8")

In [127]:
info_data(panel_modelo_final)

El dataset consta de  (199716, 62)  (FILAS/COLUMNAS)

Información del dataset: 

<class 'pandas.DataFrame'>
RangeIndex: 199716 entries, 0 to 199715
Data columns (total 62 columns):
 #   Column                                   Non-Null Count   Dtype    
---  ------                                   --------------   -----    
 0   codigo_municipio                         199716 non-null  str      
 1   mes                                      199716 non-null  int64    
 2   precipitacion_media_mensual_mm           199360 non-null  float64  
 3   anio                                     199716 non-null  int64    
 4   periodo                                  199716 non-null  period[M]
 5   nombre_municipio                         199716 non-null  str      
 6   codigo_departamento                      199716 non-null  int64    
 7   altitud_municipio_m                      199716 non-null  int64    
 8   area_municipio_km2                       199716 non-null  float64  
 9   longitud_dr

Revisión final

In [128]:
# No duplicados por llave
panel_modelo_final.duplicated(["codigo_municipio", "anio", "mes"]).sum()

np.int64(0)

In [129]:
# Cobertura del target
panel_modelo_final[[
    "hubo_inundacion_mes",
    "hubo_inundacion_agricola_mes",
    "hectareas_afectadas_suma_mes"
]].describe()

,hubo_inundacion_mes,hubo_inundacion_agricola_mes,hectareas_afectadas_suma_mes
count,199716.000000,199716.000000,199716.000000
mean,0.005478,0.003725,1.635370
std,0.073809,0.060922,109.848259
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,1.000000,1.000000,25000.000000


In [130]:
# Años disponibles
sorted(panel_modelo_final["anio"].unique())

[np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024)]

In [131]:
# Revisión rápida de EVA
panel_modelo_final.groupby("anio")[[
    "arroz_area_cosechada_ha",
    "maiz_area_cosechada_ha",
    "indicador_eva_observada_fuente"
]].mean()

,arroz_area_cosechada_ha,maiz_area_cosechada_ha,indicador_eva_observada_fuente
anio,,,
2010,0.000000,0.000000,0.0
2011,0.000000,0.000000,0.0
2012,0.000000,0.000000,0.0
2013,0.000000,0.000000,0.0
2014,0.000000,0.000000,0.0
2015,0.000000,0.000000,0.0
2016,0.000000,0.000000,0.0
2017,0.000000,0.000000,0.0
2018,0.000000,0.000000,0.0


## Guardar/exportar datasets

In [132]:
panel_exportar = panel_modelo_final.copy()
panel_exportar["periodo"] = panel_exportar["periodo"].astype(str)

panel_modelo_2010_2024_base = panel_exportar.copy()
panel_modelo_2019_2024_con_eva = panel_exportar[
    panel_exportar["anio"] >= 2019
].copy()

base_frecuencia = panel_modelo_2010_2024_base.copy()

base_severidad = panel_modelo_2019_2024_con_eva[
    panel_modelo_2019_2024_con_eva["hubo_inundacion_agricola_mes"] == 1
].copy()

panel_modelo_2010_2024_base.to_csv("panel_modelo_2010_2024_base.csv", index=False, encoding="utf-8-sig")
panel_modelo_2019_2024_con_eva.to_csv("panel_modelo_2019_2024_con_eva.csv", index=False, encoding="utf-8-sig")
base_frecuencia.to_csv("base_frecuencia.csv", index=False, encoding="utf-8-sig")
base_severidad.to_csv("base_severidad.csv", index=False, encoding="utf-8-sig")